# Example 10.3 — Post-Optimization Analysis: Transmission Angle and Grashof

**Chapter 10: Complete Example: Step-by-Step Trajectory Synthesis**

Analyzes the rounded optimum of Example 10.2 over a full crank revolution: Grashof condition, transmission angle, assemblability and rocker continuity.

Only NumPy is used for the computation (`from numpy import *`); Matplotlib is used only for the figures.

In [1]:
import matplotlib.pyplot as plt
from numpy import *

def drop_trajectory(n=20):
    t = linspace(0, 2 * pi, n, endpoint=False)
    f = 1 - 0.5 * sin(t / 2)
    return 100 + 50 * cos(t) * f, 100 + 50 * sin(t) * f

def analyze(r1, r2, r3, r4, t2):
    A = 2 * r4 * (r1 - r2 * cos(t2))
    B = -2 * r2 * r4 * sin(t2)
    C = r1**2 + r2**2 + r4**2 - r3**2 - 2 * r1 * r2 * cos(t2)
    disc = A**2 + B**2 - C**2
    root = sqrt(where(disc >= 0, disc, 0.0))
    t4 = where(disc >= 0, 2 * arctan2(-B - root, C - A), nan)
    t3 = arctan2(r4 * sin(t4) - r2 * sin(t2),
                 r1 + r4 * cos(t4) - r2 * cos(t2))
    return t3, t4

# Optimal design of Example 10.2 (seed 42), rounded to 2 dec.
r1, r2, r3, r4 = 120.85, 32.53, 127.28, 69.83
xp, yp, theta20 = -0.97, 6.32, 1.37
x0, y0 = 108.62, 95.53

# 1) Grashof classification
L = sort(array([r1, r2, r3, r4]))
sl, pq = L[0] + L[3], L[1] + L[2]
print(f'Grashof: s + l = {sl:.2f} mm, p + q = {pq:.2f} mm ->',
      'satisfied' if sl < pq else 'violated')
print(f'Shortest link is the input crank r2: {r2 == L[0]}'
      '  (crank-rocker)')

# 2) Trajectory error at the 20 precision points
xd, yd = drop_trajectory(20)
t2 = radians(theta20) + linspace(0, 2 * pi, 20,
                                 endpoint=False)
t3, t4 = analyze(r1, r2, r3, r4, t2)
xa = x0 + r2 * cos(t2) + xp * cos(t3) - yp * sin(t3)
ya = y0 + r2 * sin(t2) + xp * sin(t3) + yp * cos(t3)
e = sqrt((xa - xd)**2 + (ya - yd)**2)
res = abs(r2 * exp(1j * t2) + r3 * exp(1j * t3)
          - r1 - r4 * exp(1j * t4))
print(f'Loop-closure residual: {res.max():.2e} mm')
print(f'Trajectory error: mean = {e.mean():.2f} mm')
print(f'                  max = {e.max():.2f} mm '
      f'(point {argmax(e)}), min = {e.min():.2f} mm')

# 3) Transmission angle over one crank revolution
th = linspace(0, 2 * pi, 360, endpoint=False)
_, th4 = analyze(r1, r2, r3, r4, th)
d2 = r1**2 + r2**2 - 2 * r1 * r2 * cos(th)
mu = degrees(arccos(clip((r3**2 + r4**2 - d2)
                         / (2 * r3 * r4), -1, 1)))
print(f'Transmission angle: min = {mu.min():.2f} deg at '
      f'theta2 = {degrees(th[argmin(mu)]):.0f} deg')
print(f'                    max = {mu.max():.2f} deg at '
      f'theta2 = {degrees(th[argmax(mu)]):.0f} deg')
print(f'                    mean = {mu.mean():.2f} deg')
inside = mean((mu >= 40) & (mu <= 140))
print(f'Samples with 40 deg <= mu <= 140 deg: '
      f'{100 * inside:.1f} %')

# 4) Assembly-branch continuity
jump = abs(diff(unwrap(append(th4, th4[0]))))
print(f'Assemblable at all samples: {not isnan(th4).any()}')
print(f'Largest rocker step: {degrees(jump.max()):.2f} deg '
      f'per 1 deg of crank')

fig, ax = plt.subplots(1, 2, figsize=(6.4, 2.7))
ax[0].axhspan(40, 140, color='tab:green', alpha=0.15,
              label='Admissible zone')
ax[0].plot(degrees(th), mu, 'k', lw=1.0)
ax[0].set_xlim(0, 360); ax[0].set_ylim(0, 180)
ax[0].set_xlabel(r'Crank angle $\theta_2$ (deg)')
ax[0].set_ylabel(r'Transmission angle $\mu$ (deg)')
ax[0].legend(fontsize=7, loc='lower right')
ax[1].bar(range(20), e, color='tab:blue')
ax[1].axhline(e.mean(), color='k', ls='--', lw=0.8,
              label='Mean error')
ax[1].set_xlabel('Precision point i')
ax[1].set_ylabel('Error $e_i$ (mm)')
ax[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

Grashof: s + l = 159.81 mm, p + q = 190.68 mm -> satisfied
Shortest link is the input crank r2: True  (crank-rocker)
Loop-closure residual: 3.99e-14 mm
Trajectory error: mean = 5.35 mm
                  max = 13.02 mm (point 0), min = 2.64 mm
Transmission angle: min = 41.68 deg at theta2 = 0 deg
                    max = 97.92 deg at theta2 = 180 deg
                    mean = 71.11 deg
Samples with 40 deg <= mu <= 140 deg: 100.0 %
Assemblable at all samples: True
Largest rocker step: 0.61 deg per 1 deg of crank